# 81城有向链路汇总

- 写入日期：2026-10-17
- 修改日期：2026-10-20
- 修改内容：在第二步排除负时间间隔及起终点相同的接续记录，另存三份筛选接续；更新六份链路数据和运行检查记录。

## 1 说明

输入为步骤01的2021—2023年城市接续CSV和固定81城名单，时间范围为各年5月1日至6月25日。三年分开，不改原始GNSS、第一步代码或第一步文件。

先从每年的全部城市接续中删除以下行：

- 时间间隔小于0：例如A市10:00结束，B市09:30开始，间隔−0.5小时。等于0正常保留；没有新增最大时间间隔阈值。
- 起终点城市相同：例如A→A、C→C。两类条件同时满足时只删一次，删除明细列出两个原因。

只删接续记录，不删访问位置，不重新拼接城市顺序，不重算已有边级和权重。A₁-B₂-C₃-D₄-C₅-E₆-A₇中排除A₁→A₇、C₃→C₅；C₃→E₆中间仍经过D₄、C₅，边级3、权重0.25；C₅→E₆边级1、权重1。若时间符合且C、E在81城，C→E主版本贡献1.25，备用版本贡献2。D→C、E→A这样的直接回流正常保留，只要符合上述条件。

三份“筛选接续”保留全部轨迹城市及原字段。汇总链路时才保留两端均在81城的记录，按年份、起点、终点汇总：主版本求衰减权重和，备用版本每行计1。同一农机重复贡献逐行计入；A→B和B→A分开。多级关系不代表直接移动，结果不证明因果。

第一步统计图及回流路径仍描述未筛选的第一步数据，本Notebook不重建这些图，也不把它们当作第二步筛选后统计。必要字段缺失或文件指纹不符时停止，不填补。


## 2 参数

依赖Python、pandas、numpy、IPython。可从项目根目录或本Notebook所在目录运行。默认不覆盖已有结果；明确重跑时将下方ALLOW_OVERWRITE改为True，或设置环境变量CITY_LINK_ALLOW_OVERWRITE=1。

按用户要求，普通重跑不自动向Archive复制数据。此次代码修改已单独备份至Archive/2026-10-20_第二步筛选实施前；以后修改代码或方案时再按需备份。


In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import os
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start_path):
    for candidate in [Path(start_path).resolve(), *Path(start_path).resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'Data').is_dir():
            return candidate
    raise FileNotFoundError('没有找到项目根目录。')

def calculate_sha256(file_path):
    digest = hashlib.sha256()
    with open(file_path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_csv_atomic(dataframe, target_path):
    # 先完整写入临时文件，再替换目标文件，避免留下半份CSV。
    temporary_path = target_path.with_name(f'.{target_path.stem}_{run_id}.tmp.csv')
    try:
        dataframe.to_csv(temporary_path, index=False, encoding='utf-8-sig')
        os.replace(temporary_path, target_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

project_root = find_project_root(Path.cwd())
analysis_years = [2021, 2022, 2023]
ALLOW_OVERWRITE = os.getenv('CITY_LINK_ALLOW_OVERWRITE', '0') == '1'
edge_relative_paths = {year: f'Data/03_有向边数据/{year}年农机城市接续.csv' for year in analysis_years}
edge_files = {year: project_root / path for year, path in edge_relative_paths.items()}
city_list_file = project_root / 'Data/02_筛选后81城数据/citys_2023_1.csv'
manifest_file = project_root / 'Data/data_manifest.csv'
output_data_dir = project_root / 'Data/04_城市链路数据'
check_result_dir = project_root / 'Results/01_链路数据检查'
notebook_relative_path = 'Code/02_构建链路数据/build_city_links.ipynb'
cleaned_output_files = {year: output_data_dir / f'{year}年筛选接续.csv' for year in analysis_years}
weighted_output_files = {year: output_data_dir / f'{year}年衰减链路.csv' for year in analysis_years}
count_output_files = {year: output_data_dir / f'{year}年计数链路.csv' for year in analysis_years}
deletion_output_file = check_result_dir / '接续删除记录.csv'
check_output_file = check_result_dir / '链路数据检查.csv'
output_files = [*cleaned_output_files.values(), *weighted_output_files.values(),
                *count_output_files.values(), deletion_output_file, check_output_file]
run_time = datetime.now().astimezone()
run_id = run_time.strftime('%Y%m%dT%H%M%S%z')
existing_outputs = [path for path in output_files if path.exists()]
if existing_outputs and not ALLOW_OVERWRITE:
    raise FileExistsError('输出已经存在；确认重跑时设置ALLOW_OVERWRITE=True或CITY_LINK_ALLOW_OVERWRITE=1。')
print('项目根目录：', project_root)
print('运行编号：', run_id)


项目根目录： C:\Users\Administrator\Desktop\W
运行编号： 20261020T133510+0800


## 3 读取

固定81城名单已在数据字典登记为正式名单。年度接续输入必须在数据清单中标为正式，且行数、大小和文件指纹相符。文件指纹用于判断内容有没有变化，不代表数据质量。读取原列为文本，以便筛选输出原样保留其数值、编号和时间表达；计算时另建英文列。


In [2]:
for input_path in [manifest_file, city_list_file, *edge_files.values()]:
    if not input_path.is_file():
        raise FileNotFoundError(f'缺少必要输入：{input_path}')
manifest = pd.read_csv(manifest_file, encoding='utf-8-sig', dtype='string')
assert not manifest['relative_path'].duplicated().any(), '数据清单路径重复。'
manifest_by_path = manifest.set_index('relative_path')
city_list = pd.read_csv(city_list_file, encoding='utf-8-sig', dtype='string')
assert city_list.columns.tolist() == ['city'], '城市名单字段应为city。'
city_list['city'] = city_list['city'].str.strip()
assert not city_list['city'].isna().any() and city_list['city'].ne('').all(), '名单城市缺失。'
assert len(city_list) == city_list['city'].nunique() == 81, '名单应为81个唯一城市。'
city_set = set(city_list['city'])
city_list_sha256 = calculate_sha256(city_list_file)
input_column_map = {
    '边记录编号': 'edge_id', '年份': 'year', '农机编号': 'did',
    '链段编号': 'chain_segment', '起点访问序号': 'source_visit_order',
    '终点访问序号': 'target_visit_order', '起点城市': 'source_city',
    '终点城市': 'target_city', '边级': 'edge_level', '跨越城市数': 'skipped_visit_count',
    '衰减权重': 'edge_weight', '起点离开时间': 'source_exit_time',
    '终点进入时间': 'target_entry_time', '时间间隔小时': 'gap_hours',
    '是否直接回流边': 'is_direct_return_edge', '是否同城端点': 'is_same_city_endpoint',
    '是否时间冲突': 'has_time_conflict',
}

def read_year_records(year):
    relative_path = edge_relative_paths[year]
    assert relative_path in manifest_by_path.index, f'清单未登记{relative_path}。'
    manifest_row = manifest_by_path.loc[relative_path]
    assert manifest_row['formal_status'] == '正式', f'{relative_path}不是正式输入。'
    input_sha256 = calculate_sha256(edge_files[year])
    assert input_sha256 == manifest_row['sha256'], f'{year}年输入文件指纹不符。'
    assert edge_files[year].stat().st_size == int(manifest_row['file_size_bytes']), '输入文件大小不符。'
    raw_records = pd.read_csv(edge_files[year], encoding='utf-8-sig', dtype='string', keep_default_na=False)
    assert len(raw_records) == int(manifest_row['row_count']), f'{year}年输入行数不符。'
    assert set(input_column_map).issubset(raw_records.columns), f'{year}年缺少必要字段。'
    assert not raw_records.isna().any().any() and not raw_records.eq('').any().any(), '输入存在缺失，不填补。'
    records = raw_records[list(input_column_map)].rename(columns=input_column_map).copy()
    for column in ['year', 'chain_segment', 'source_visit_order', 'target_visit_order',
                   'edge_level', 'skipped_visit_count', 'edge_weight', 'gap_hours']:
        records[column] = pd.to_numeric(records[column], errors='raise')
        assert np.isfinite(records[column]).all(), f'{column}有非有限数值。'
    for column in ['year', 'chain_segment', 'source_visit_order', 'target_visit_order', 'edge_level', 'skipped_visit_count']:
        assert records[column].eq(np.floor(records[column])).all(), f'{column}必须为整数。'
    for column in ['source_city', 'target_city', 'did']:
        records[column] = records[column].str.strip()
        assert records[column].ne('').all(), f'{column}为空。'
    for column in ['is_direct_return_edge', 'is_same_city_endpoint', 'has_time_conflict']:
        values = records[column].str.lower()
        assert values.isin(['true', 'false']).all(), f'{column}不是有效布尔值。'
        records[column] = values.eq('true')
    assert records['year'].eq(year).all(), '输入含其他年份。'
    assert records['source_visit_order'].ge(1).all() and records['chain_segment'].ge(1).all()
    assert not records['edge_id'].duplicated().any(), '边记录编号重复。'
    assert not records.duplicated(['year', 'did', 'chain_segment', 'source_visit_order', 'target_visit_order']).any(), '访问对重复。'
    assert records['edge_level'].ge(1).all()
    assert records['edge_level'].eq(records['target_visit_order'] - records['source_visit_order']).all()
    assert records['skipped_visit_count'].eq(records['edge_level'] - 1).all()
    assert np.allclose(records['edge_weight'], np.power(0.5, records['edge_level'] - 1), rtol=0, atol=1e-12)
    source_exit_time = pd.to_datetime(records['source_exit_time'], errors='raise')
    target_entry_time = pd.to_datetime(records['target_entry_time'], errors='raise')
    calculated_gap = (target_entry_time - source_exit_time).dt.total_seconds() / 3600
    assert np.allclose(records['gap_hours'], calculated_gap, rtol=0, atol=1e-9), '时间间隔与两端时间不符。'
    assert records['has_time_conflict'].eq(records['gap_hours'].lt(0)).all(), '时间标记不符。'
    assert records['is_same_city_endpoint'].eq(records['source_city'].eq(records['target_city'])).all(), '城市标记不符。'
    return raw_records, records, input_sha256

print('输入路径及81城名单检查通过；逐年读取和核对在下一单元执行。')


输入路径及81城名单检查通过；逐年读取和核对在下一单元执行。


## 4 处理

先对全部城市接续逐行判断两类删除条件并保存原因，再从保留记录中选择两端均在81城的行汇总。不按城市名称去重访问，不压缩序号，不因某条边被删而连带删除其他边。


In [3]:
def get_exclusion_masks(records):
    time_conflict_mask = records['gap_hours'].lt(0)
    same_city_mask = records['source_city'].eq(records['target_city'])
    exclusion_mask = time_conflict_mask | same_city_mask
    return time_conflict_mask, same_city_mask, exclusion_mask

cleaned_outputs = {}
aggregated_internal = {}
deletion_parts = []
input_stats = []
deletion_columns = ['边记录编号', '年份', '农机编号', '链段编号', '起点访问序号', '终点访问序号',
                    '起点城市', '终点城市', '边级', '衰减权重', '起点离开时间',
                    '终点进入时间', '时间间隔小时', '来源文件']
for year in analysis_years:
    raw_records, records, input_sha256 = read_year_records(year)
    time_conflict_mask, same_city_mask, exclusion_mask = get_exclusion_masks(records)
    kept_records = records.loc[~exclusion_mask].copy()
    cleaned_outputs[year] = raw_records.loc[~exclusion_mask].copy()
    deleted_records = raw_records.loc[exclusion_mask, deletion_columns].copy()
    deletion_reason = pd.Series('', index=records.index, dtype='string')
    deletion_reason.loc[time_conflict_mask] = '时间间隔小于0'
    deletion_reason.loc[same_city_mask & ~time_conflict_mask] = '起终点城市相同'
    deletion_reason.loc[same_city_mask & time_conflict_mask] = '时间间隔小于0；起终点城市相同'
    deleted_records['删除原因'] = deletion_reason.loc[exclusion_mask]
    deleted_records['接续来源文件'] = edge_relative_paths[year]
    deletion_parts.append(deleted_records)
    in_81_mask = kept_records['source_city'].isin(city_set) & kept_records['target_city'].isin(city_set)
    in_81_records = kept_records.loc[in_81_mask].copy()
    in_81_records['is_first_level'] = in_81_records['edge_level'].eq(1).astype(int)
    in_81_records['is_multi_level'] = in_81_records['edge_level'].gt(1).astype(int)
    grouped = (
        in_81_records.groupby(['source_city', 'target_city'], sort=True, observed=True)
        .agg(decay_weight_sum=('edge_weight', 'sum'), record_count=('did', 'size'),
             machine_count=('did', 'nunique'), first_level_count=('is_first_level', 'sum'),
             multi_level_count=('is_multi_level', 'sum'), min_edge_level=('edge_level', 'min'),
             max_edge_level=('edge_level', 'max')).reset_index()
    )
    grouped['year'] = year
    grouped['directed_edge_id'] = grouped['source_city'] + '→' + grouped['target_city']
    grouped['source_file'] = cleaned_output_files[year].relative_to(project_root).as_posix()
    aggregated_internal[year] = grouped
    # 数量必须可还原；重叠删除只算一次。
    assert len(raw_records) == len(kept_records) + int(exclusion_mask.sum())
    assert int(exclusion_mask.sum()) == int(time_conflict_mask.sum() + same_city_mask.sum() - (time_conflict_mask & same_city_mask).sum())
    assert len(kept_records) == len(in_81_records) + int((~in_81_mask).sum())
    assert not kept_records['gap_hours'].lt(0).any()
    assert not kept_records['source_city'].eq(kept_records['target_city']).any()
    input_stats.append({
        'year': year, 'input_row_count': len(records),
        'time_conflict_count': int(time_conflict_mask.sum()), 'same_city_count': int(same_city_mask.sum()),
        'overlap_count': int((time_conflict_mask & same_city_mask).sum()),
        'deleted_row_count': int(exclusion_mask.sum()), 'kept_row_count': len(kept_records),
        'in_81_record_count': len(in_81_records), 'outside_81_count': int((~in_81_mask).sum()),
        'directed_pair_count': len(grouped), 'machine_count': int(in_81_records['did'].nunique()),
        'first_level_count': int(in_81_records['is_first_level'].sum()),
        'multi_level_count': int(in_81_records['is_multi_level'].sum()),
        'direct_return_count': int(kept_records['is_direct_return_edge'].sum()),
        'decay_weight_sum': float(in_81_records['edge_weight'].sum()), 'input_sha256': input_sha256,
    })
    del raw_records, records, kept_records, in_81_records
deletion_output = pd.concat(deletion_parts, ignore_index=True)
input_summary = pd.DataFrame(input_stats)
filter_summary = input_summary[['year', 'input_row_count', 'time_conflict_count', 'same_city_count',
                               'overlap_count', 'deleted_row_count', 'kept_row_count']].rename(columns={
    'year': '年份', 'input_row_count': '筛选前记录数', 'time_conflict_count': '时间间隔小于0',
    'same_city_count': '起终点相同', 'overlap_count': '同时满足两类', 'deleted_row_count': '实际删除',
    'kept_row_count': '筛选后记录数'})
display(filter_summary)


,年份,筛选前记录数,时间间隔小于0,起终点相同,同时满足两类,实际删除,筛选后记录数
0,2021,110458,0,7850,0,7850,102608
1,2022,275055,7,25744,2,25749,249306
2,2023,500521,25,42831,5,42851,457670


In [4]:
output_column_map = {
    'year': '年份', 'directed_edge_id': '有向边编号', 'source_city': '起点城市', 'target_city': '终点城市',
    'link_weight': '链路权重', 'weight_method': '权重口径', 'record_count': '接续记录数',
    'machine_count': '涉及农机数', 'first_level_count': '一级边记录数',
    'multi_level_count': '二级及以上边记录数', 'min_edge_level': '最小边级',
    'max_edge_level': '最大边级', 'source_file': '来源文件',
}
output_column_order = list(output_column_map.values())

def make_output_table(grouped_data, weight_method):
    output_data = grouped_data.copy()
    output_data['link_weight'] = (output_data['decay_weight_sum'] if weight_method == '衰减权重求和'
                                  else output_data['record_count'])
    output_data['weight_method'] = weight_method
    return output_data.rename(columns=output_column_map)[output_column_order].reset_index(drop=True)

weighted_outputs = {year: make_output_table(aggregated_internal[year], '衰减权重求和') for year in analysis_years}
count_outputs = {year: make_output_table(aggregated_internal[year], '每条接续记录计1') for year in analysis_years}
output_summary = pd.DataFrame([
    {'年份': stats['year'], '81城内接续记录数': stats['in_81_record_count'],
     '有向城市对数': stats['directed_pair_count'], '衰减权重合计': stats['decay_weight_sum'],
     '计数权重合计': stats['in_81_record_count']}
    for stats in input_stats
])
display(output_summary.round({'衰减权重合计': 6}))


,年份,81城内接续记录数,有向城市对数,衰减权重合计,计数权重合计
0,2021,89148,3201,48756.632278,89148
1,2022,232420,3572,125617.708141,232420
2,2023,400144,4267,205360.207628,400144


## 5 检查

检查筛选前后数量、两类重叠、城市范围、主键、原访问序号与权重、两版本城市对和权重合计。下面的独立小例子还检查重复城市、直接回流、零时间间隔及两类条件重叠。本任务是数据处理，不新增图表。


In [5]:
for year in analysis_years:
    grouped = aggregated_internal[year]
    weighted_data = weighted_outputs[year]
    count_data = count_outputs[year]
    stats = input_summary.loc[input_summary['year'].eq(year)].iloc[0]
    cleaned = cleaned_outputs[year]
    assert not grouped[['source_city', 'target_city']].duplicated().any()
    assert grouped['source_city'].ne(grouped['target_city']).all()
    assert grouped['source_city'].isin(city_set).all() and grouped['target_city'].isin(city_set).all()
    assert not weighted_data.isna().any().any() and not count_data.isna().any().any()
    assert grouped['record_count'].eq(grouped['first_level_count'] + grouped['multi_level_count']).all()
    assert grouped['machine_count'].le(grouped['record_count']).all()
    assert grouped['min_edge_level'].ge(1).all()
    assert grouped['max_edge_level'].ge(grouped['min_edge_level']).all()
    assert weighted_data[['年份', '起点城市', '终点城市']].equals(count_data[['年份', '起点城市', '终点城市']])
    assert (weighted_data['链路权重'] > 0).all()
    assert weighted_data['链路权重'].le(weighted_data['接续记录数'] + 1e-12).all()
    assert count_data['链路权重'].eq(count_data['接续记录数']).all()
    assert int(count_data['链路权重'].sum()) == stats['in_81_record_count']
    assert np.isclose(weighted_data['链路权重'].sum(), stats['decay_weight_sum'], rtol=0, atol=1e-9)
    assert len(cleaned) == stats['kept_row_count']
    # 不根据保留下来的一级边还原路径，直接核对原来的访问位置。
    source_order = pd.to_numeric(cleaned['起点访问序号'], errors='raise')
    target_order = pd.to_numeric(cleaned['终点访问序号'], errors='raise')
    levels = pd.to_numeric(cleaned['边级'], errors='raise')
    assert levels.eq(target_order - source_order).all()
    assert np.allclose(pd.to_numeric(cleaned['衰减权重']), np.power(0.5, levels - 1), rtol=0, atol=1e-12)
assert not deletion_output[['年份', '边记录编号']].duplicated().any()
assert len(deletion_output) == int(input_summary['deleted_row_count'].sum())

# 检查用户给出的重复城市例子，时间统一有效；直接调用正式筛选函数。
test_cities = ['A', 'B', 'C', 'D', 'C', 'E', 'A']
test_records = pd.DataFrame([
    {'source_city': source_city, 'target_city': target_city, 'source_visit_order': source_index + 1,
     'target_visit_order': target_index + 1, 'gap_hours': 0,
     'edge_level': target_index - source_index, 'edge_weight': 0.5 ** (target_index - source_index - 1)}
    for source_index, source_city in enumerate(test_cities)
    for target_index, target_city in enumerate(test_cities) if target_index > source_index
])
test_time_mask, test_city_mask, test_exclusion = get_exclusion_masks(test_records)
test_kept = test_records.loc[~test_exclusion]
assert len(test_records) == 21 and len(test_kept) == 19
assert not test_kept['source_city'].eq(test_kept['target_city']).any()
test_c_to_e = test_kept.loc[test_kept['source_city'].eq('C') & test_kept['target_city'].eq('E')]
assert test_c_to_e['edge_weight'].tolist() == [0.25, 1.0]
assert test_c_to_e['edge_weight'].sum() == 1.25 and len(test_c_to_e) == 2
assert ((test_kept['source_visit_order'] == 4) & (test_kept['target_visit_order'] == 5)).any()
assert ((test_kept['source_visit_order'] == 6) & (test_kept['target_visit_order'] == 7)).any()
boundary_records = pd.DataFrame({'source_city': ['A', 'A', 'A', 'A'], 'target_city': ['B', 'B', 'A', 'A'],
                                 'gap_hours': [-0.5, 0, 1, -1]})
boundary_time, boundary_city, boundary_exclusion = get_exclusion_masks(boundary_records)
assert boundary_time.tolist() == [True, False, False, True]
assert boundary_city.tolist() == [False, False, True, True]
assert boundary_exclusion.tolist() == [True, False, True, True]
print('全部写出前检查通过；重复城市和零间隔等小例子检查通过。')


全部写出前检查通过；重复城市和零间隔等小例子检查通过。


## 6 输出

输出三份筛选接续、三份衰减链路、三份计数链路，以及接续删除记录和链路数据检查。六份链路不再含“同城端点记录数”“时间冲突记录数”“是否同城边”三个旧说明列，删除原因集中放在删除记录中。

筛选接续原字段完整保留，因此原来的两个异常标记均为false；不再作为单独研究类型。数据清单由本Notebook按实际输出同步更新，不修改任何研究规则文档。


In [6]:
output_data_dir.mkdir(parents=True, exist_ok=True)
check_result_dir.mkdir(parents=True, exist_ok=True)
for year in analysis_years:
    for dataframe, output_path in [(cleaned_outputs[year], cleaned_output_files[year]),
                                    (weighted_outputs[year], weighted_output_files[year]),
                                    (count_outputs[year], count_output_files[year])]:
        write_csv_atomic(dataframe, output_path)
        readback = pd.read_csv(output_path, encoding='utf-8-sig', dtype='string', keep_default_na=False)
        pd.testing.assert_frame_equal(readback, dataframe.astype('string').reset_index(drop=True), check_dtype=False)
write_csv_atomic(deletion_output, deletion_output_file)
deletion_readback = pd.read_csv(deletion_output_file, encoding='utf-8-sig', dtype='string', keep_default_na=False)
pd.testing.assert_frame_equal(deletion_readback, deletion_output.astype('string').reset_index(drop=True), check_dtype=False)

check_rows = []
def add_check(year, metric_group, metric, value, unit, note):
    check_rows.append({'运行编号': run_id, '运行时间': run_time.isoformat(timespec='seconds'),
                       '年份': year, '指标组': metric_group, '指标': metric, '数值': value,
                       '单位': unit, '检查状态': '通过', '说明': note})
for stats in input_stats:
    year = stats['year']
    for metric, value, note in [
        ('年度接续记录数', stats['input_row_count'], '与输入清单行数一致；筛选前、全部城市'),
        ('时间间隔小于0的记录数', stats['time_conflict_count'], '删除；等于0保留'),
        ('起终点城市相同的记录数', stats['same_city_count'], '只删边，不删城市访问'),
        ('同时满足两类删除条件的记录数', stats['overlap_count'], '两类计数均含此部分；实际删除只计一次'),
        ('实际删除记录数', stats['deleted_row_count'], '两类数量相加减去重叠数量'),
        ('筛选后全部城市接续记录数', stats['kept_row_count'], '保留原访问位置、边级及权重'),
        ('两端都在81城的记录数', stats['in_81_record_count'], '从筛选后接续选择，进入链路汇总'),
        ('筛选后至少一端不在81城的记录数', stats['outside_81_count'], '保留在筛选接续中，不进入81城链路'),
        ('有向城市对数', stats['directed_pair_count'], 'A到B和B到A分开，不含同城对'),
        ('涉及农机数', stats['machine_count'], '81城内记录中不同农机数'),
        ('一级边记录数', stats['first_level_count'], '81城内原边级等于1'),
        ('二级及以上边记录数', stats['multi_level_count'], '81城内原边级大于1'),
        ('筛选后全部城市直接回流边数', stats['direct_return_count'], '沿用完整原序列标记，不重建访问顺序'),
    ]:
        add_check(year, '筛选与汇总', metric, int(value), '条或台', note)
    add_check(year, '权重', '衰减权重合计', stats['decay_weight_sum'], '权重', '81城内筛选后接续的衰减权重之和')
    add_check(year, '权重', '计数权重合计', stats['in_81_record_count'], '权重', '81城内筛选后每条记录计1')
    assert calculate_sha256(edge_files[year]) == stats['input_sha256'], '输入在运行期间改变。'
    add_check(year, '文件核对', '输入文件指纹', stats['input_sha256'], '文件指纹', edge_relative_paths[year])
    for label, path, data in [('筛选接续', cleaned_output_files[year], cleaned_outputs[year]),
                             ('衰减链路', weighted_output_files[year], weighted_outputs[year]),
                             ('计数链路', count_output_files[year], count_outputs[year])]:
        add_check(year, '文件核对', f'{label}输出行数', len(data), '行', path.relative_to(project_root).as_posix())
        add_check(year, '文件核对', f'{label}文件指纹', calculate_sha256(path), '文件指纹', path.relative_to(project_root).as_posix())
assert calculate_sha256(city_list_file) == city_list_sha256, '城市名单在运行期间改变。'
add_check('全部', '文件核对', '81城名单文件指纹', city_list_sha256, '文件指纹', city_list_file.relative_to(project_root).as_posix())
add_check('全部', '文件核对', '接续删除记录行数', len(deletion_output), '行', deletion_output_file.relative_to(project_root).as_posix())
add_check('全部', '文件核对', '接续删除记录文件指纹', calculate_sha256(deletion_output_file), '文件指纹', '含两类原因重叠的行，但每条边只写一次')
add_check('全部', '来源', '生成Notebook', notebook_relative_path, '路径', '全部处理代码均在此Notebook')
add_check('全部', '规则检查', '重复城市和边界例子', '通过', '结果', '首次C到E为0.25；第二次为1；零间隔保留；重叠删除一次')
check_output = pd.DataFrame(check_rows)
write_csv_atomic(check_output, check_output_file)
check_readback = pd.read_csv(check_output_file, encoding='utf-8-sig', dtype='string', keep_default_na=False)
pd.testing.assert_frame_equal(check_readback, check_output.astype('string'), check_dtype=False)
for path in output_files:
    with open(path, 'rb') as handle:
        assert handle.read(3) == b'\xef\xbb\xbf', f'{path.name}不是UTF-8-SIG编码。'


In [7]:
# 清单只更新本步骤输出；第一步输入仍按原文件登记，防止混淆。
manifest_updates = []
def register_output(file_id, path, year, row_count, grain, upstream, purpose, notes, role='formal_intermediate'):
    manifest_updates.append({
        'file_id': file_id, 'relative_path': path.relative_to(project_root).as_posix(), 'file_name': path.name,
        'file_role': role, 'formal_status': '正式', 'read_only': 'false', 'year': str(year),
        'time_range': f'{year}-05-01至{year}-06-25' if str(year).isdigit() else '各年05-01至06-25',
        'data_grain': grain, 'row_count': str(row_count), 'file_size_bytes': str(path.stat().st_size),
        'sha256': calculate_sha256(path), 'encoding': 'UTF-8-SIG', 'source_or_generator': notebook_relative_path,
        'upstream': upstream, 'purpose': purpose, 'notes': notes,
    })
for stats in input_stats:
    year = stats['year']
    cleaned_relative = cleaned_output_files[year].relative_to(project_root).as_posix()
    register_output(f'CITY_EDGE_FILTERED_{year}', cleaned_output_files[year], year, stats['kept_row_count'],
                    '一行一台农机的一对保留的前后城市访问位置', edge_relative_paths[year],
                    '第二步筛选后的全部城市接续；供链路汇总和后续明细分析使用',
                    '删除负时间间隔及起终点相同行；保留全部轨迹城市和原访问序号、边级、权重')
    for version, path, notes in [
        ('DECAY', weighted_output_files[year], '主版本；同方向城市对衰减权重求和'),
        ('COUNT', count_output_files[year], '备用版本；每条接续记录计1')]:
        register_output(f'CITY_LINK_{version}_{year}', path, year, stats['directed_pair_count'],
                        '一行一个已出现的81城内不同城市有向对',
                        cleaned_relative + '；Data/02_筛选后81城数据/citys_2023_1.csv',
                        '供基线链路构建或年度比较使用' if version == 'DECAY' else '供备用权重比较使用',
                        notes + '；已排除负时间间隔和同城边')
register_output('CITY_EDGE_DELETION', deletion_output_file, '2021-2023', len(deletion_output),
                '一行一条被排除的接续记录', '三份步骤01年度农机城市接续CSV',
                '核对删除原因及原边记录位置', '重叠原因同一行列明，只删除一次', 'quality_output')
register_output('CITY_LINK_CHECK', check_output_file, '2021-2023', len(check_output),
                '一行一个年份和检查项目', '三份步骤01接续；固定81城名单；本步骤九份数据及删除记录',
                '保存文件核对、筛选前后数量和汇总检查结果',
                '文件核对结果存放此处，不写入正式决策记录', 'quality_output')
updates = pd.DataFrame(manifest_updates, columns=manifest.columns)
manifest_updated = pd.concat([manifest.loc[~manifest['file_id'].isin(updates['file_id'])], updates], ignore_index=True)
assert not manifest_updated['file_id'].duplicated().any()
assert not manifest_updated['relative_path'].duplicated().any()
# 保持Data文件夹在前、Results在后；目录内按年份和文件名排列。
manifest_updated = manifest_updated.sort_values('relative_path', kind='stable').reset_index(drop=True)
write_csv_atomic(manifest_updated, manifest_file)
manifest_readback = pd.read_csv(manifest_file, encoding='utf-8-sig', dtype='string')
pd.testing.assert_frame_equal(manifest_readback.fillna(''), manifest_updated.fillna('').astype('string'), check_dtype=False)
display(filter_summary)
display(output_summary.round({'衰减权重合计': 6}))
print('从头到尾运行完成；11份输出已读回核对，数据清单已更新。')
print('原始GNSS、第一步接续文件和第一步代码均未修改。')
print('文件核对和删除记录：', check_result_dir)


,年份,筛选前记录数,时间间隔小于0,起终点相同,同时满足两类,实际删除,筛选后记录数
0,2021,110458,0,7850,0,7850,102608
1,2022,275055,7,25744,2,25749,249306
2,2023,500521,25,42831,5,42851,457670


,年份,81城内接续记录数,有向城市对数,衰减权重合计,计数权重合计
0,2021,89148,3201,48756.632278,89148
1,2022,232420,3572,125617.708141,232420
2,2023,400144,4267,205360.207628,400144


从头到尾运行完成；11份输出已读回核对，数据清单已更新。
原始GNSS、第一步接续文件和第一步代码均未修改。
文件核对和删除记录： C:\Users\Administrator\Desktop\W\Results\01_链路数据检查
